In [0]:
from pyspark.sql import functions as F

In [0]:
sql_server_name = "quant-cloud-server"
database_name = "my-database"
port = 1433

jdbc_url = f"""jdbc:sqlserver://{sql_server_name}.database.windows.net:{port};database={database_name};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"""

user = dbutils.secrets.get(scope="sc-rainbow-batch-04", key="sql-server-user")
password = dbutils.secrets.get(scope="sc-rainbow-batch-04", key="sql-server-password")

In [0]:
# dbutils.fs.mounts()

In [0]:
# test

# df = (
#     spark.read.format("csv")
#     .option("header", True)
#     .option("inferSchema", True)
#     .load("/mnt/rainbow-container/bihar_election_results.csv")
# )

# df.display()

In [0]:
# dbutils.fs.ls("/mnt/rainbow-container")

In [0]:

def get_csv_files(root_path: str):
    csv_files = []
    for file in dbutils.fs.ls(root_path):
        if (file.name).endswith(".csv"):
            csv_files.append(file.path)
    
    return csv_files

In [0]:
# print(get_csv_files("/mnt/rainbow-container"))

In [0]:
def write_df_into_sql_db():
    paths = get_csv_files("/mnt/rainbow-container")
    for path in paths:
        df = (
            spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .load(path)
        )
        file_name = path.split("/")[-1].replace(".csv", "").replace(" ", "_").lower()
        # print(file_name)
        (
            df.write.format("jdbc")
            .mode("overwrite")
            .option("url", jdbc_url)
            .option("user", user)
            .option("password", password)
            .option("dbtable", file_name)
            .save()
        )

In [0]:
write_df_into_sql_db()